### Bloco 1: Instalar dependências necessárias (incluindo statsmodels para ARIMA)

In [25]:
!pip install -q fastapi uvicorn scikit-learn joblib tensorflow pydantic nest-asyncio pyngrok statsmodels pandas requests

### Bloco 2: Montar o Google Drive e verificar dimensões do modelo e scalers

In [26]:
import os, joblib
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
CAMINHO_PROJETO = '/content/drive/MyDrive/Projeto_lstm_iot_Colab'
os.chdir(CAMINHO_PROJETO)

model_check = load_model('modelo_lstm_multivariado.keras')
scaler_x_check = joblib.load('scaler_x.pkl')
print('Input shape do modelo:', model_check.input_shape)
print('Features esperadas pelo scaler_x:', scaler_x_check.n_features_in_)
print('Mínimos (data_min_):', scaler_x_check.data_min_)
print('Máximos (data_max_):', scaler_x_check.data_max_)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input shape do modelo: (None, 10, 6)
Features esperadas pelo scaler_x: 6
Mínimos (data_min_): [ 0.00000000e+00  0.00000000e+00  1.16800000e+02 -3.17743075e+06
  1.67000000e-01  0.00000000e+00]
Máximos (data_max_): [1.2450000e+02 7.2150000e+00 1.3860001e+02 9.3100000e-01 1.1646000e+01
 8.8000000e-02]


### Bloco 3: Subir a API FastAPI (LSTM Multivariado + ARIMA + Histórico) com Ngrok

In [28]:
import os, json, joblib, nest_asyncio, requests, uvicorn
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from tensorflow.keras.models import load_model
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FIREBASE_URL = "https://consumo-conciente-default-rtdb.firebaseio.com"
CAMINHO_NO_FIREBASE = "/leituras"

app = FastAPI(title="API LSTM Multivariado & ARIMA - IFES")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

print("Carregando modelo e scalers...")
model = load_model("modelo_lstm_multivariado.keras")
scaler_x = joblib.load("scaler_x.pkl")
scaler_y = joblib.load("scaler_y.pkl")

LOOK_BACK = model.input_shape[1]   # 10
N_FEATURES = model.input_shape[2]  # 6
print(f"Modelo carregado! Formato: ({LOOK_BACK} passos, {N_FEATURES} features)")

def buscar_dados_firebase(limite=150):
    url = f"{FIREBASE_URL}{CAMINHO_NO_FIREBASE}.json"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    dados_brutos = resp.json()
    if not dados_brutos or not isinstance(dados_brutos, dict):
        raise ValueError("Nenhum dado encontrado no Firebase.")
    registros = []
    itens_ordenados = sorted(dados_brutos.items(), key=lambda x: str(x[0]))
    consumo_anterior = None
    for chave, item in itens_ordenados:
        if not isinstance(item, dict):
            continue
        voltagem = float(item.get("voltagem", item.get("tensao", item.get("voltage", 127.0))))
        corrente = float(item.get("corrente", item.get("current", 0.0)))
        potencia = float(item.get("potencia", item.get("power", 0.0)))
        consumo = float(item.get("consumo_total_pzem", item.get("energia", item.get("consumo", 0.0))))
        fp = float(item.get("fator_potencia", item.get("fp", item.get("pf", 0.95))))
        freq = float(item.get("frequencia", item.get("freq", 60.0)))
        delta_energia = float(item["delta_energia"]) if "delta_energia" in item else (max(0.0, consumo - consumo_anterior) if consumo_anterior is not None else 0.0)
        consumo_anterior = consumo
        registros.append({
            "timestamp": str(item.get("timestamp", item.get("data_hora", chave))),
            "voltagem": voltagem,
            "corrente": corrente,
            "potencia": potencia,
            "consumo_total_pzem": consumo,
            "delta_energia": delta_energia,
            "fator_potencia": fp,
            "frequencia": freq
        })
    if len(registros) > limite:
        registros = registros[-limite:]
    while len(registros) < LOOK_BACK:
        base = registros[0].copy() if registros else {
            "timestamp": "Inicial", "voltagem": 127.0, "corrente": 0.0,
            "potencia": 0.0, "consumo_total_pzem": 0.0, "delta_energia": 0.0,
            "fator_potencia": 1.0, "frequencia": 60.0
        }
        registros.insert(0, base)
    return registros

def extrair_matriz_features(janela):
    matriz = [
        [
            item["voltagem"],
            item["corrente"],
            item["potencia"],
            item["consumo_total_pzem"],
            item["fator_potencia"],
            item["delta_energia"]
        ][:N_FEATURES]
        for item in janela[-LOOK_BACK:]
    ]
    return np.array(matriz, dtype=float)

def rodar_previsao_lstm(janela):
    scaled_data = scaler_x.transform(extrair_matriz_features(janela))
    X_input = np.reshape(scaled_data, (1, LOOK_BACK, N_FEATURES))
    prediction_rescaled = scaler_y.inverse_transform(model.predict(X_input, verbose=0))
    return float(max(0.0, prediction_rescaled[0, 0]))

@app.get("/")
def health_check():
    return {"status": "online", "modelo": "LSTM Multivariado + ARIMA", "look_back": LOOK_BACK, "features": N_FEATURES}

@app.get("/api/status_atual")
def status_atual():
    try:
        historico = buscar_dados_firebase(limite=LOOK_BACK + 5)
        ultima = historico[-1]
        delta_prev = rodar_previsao_lstm(historico)
        return {
            "status": "sucesso",
            "leitura_atual": {
                "consumo_total_pzem": ultima["consumo_total_pzem"],
                "corrente": ultima["corrente"],
                "potencia": ultima["potencia"],
                "voltagem": ultima["voltagem"]
            },
            "previsao_proximo_passo": ultima["consumo_total_pzem"] + delta_prev,
            "delta_previsto": delta_prev
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro em status_atual: {str(e)}")

@app.get("/api/previsao_futura")
def previsao_futura(passos: int = 6):
    try:
        janela = buscar_dados_firebase(limite=LOOK_BACK + 5)
        consumo_acum = janela[-1]["consumo_total_pzem"]
        valores = []
        for _ in range(passos):
            delta_pred = rodar_previsao_lstm(janela)
            consumo_acum += delta_pred
            valores.append(round(consumo_acum, 4))
            novo = janela[-1].copy()
            novo["consumo_total_pzem"] = consumo_acum
            novo["delta_energia"] = delta_pred
            janela = janela[1:] + [novo]
        return {"status": "sucesso", "passos": passos, "valores_projetados": valores}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro na projeção: {str(e)}")

@app.get("/api/historico_treinamento")
def historico_treinamento():
    try:
        if os.path.exists("metricas_treinamento.json"):
            with open("metricas_treinamento.json", "r") as f:
                return json.load(f)
        registros = buscar_dados_firebase(limite=100)
        if len(registros) < LOOK_BACK + 10:
            raise ValueError("Poucos registros no Firebase.")
        labels, reais, preds_lstm = [], [], []
        for i in range(LOOK_BACK, len(registros)):
            labels.append(registros[i]["timestamp"][-8:])
            reais.append(round(registros[i]["delta_energia"], 5))
            preds_lstm.append(round(rodar_previsao_lstm(registros[i - LOOK_BACK : i]), 5))
        serie_arima = [r["delta_energia"] for r in registros[:LOOK_BACK]]
        try:
            preds_arima = [round(max(0.0, float(v)), 5) for v in ARIMA(serie_arima, order=(5, 1, 0)).fit().forecast(steps=len(reais))]
        except Exception:
            preds_arima = [round(float(np.mean(serie_arima)), 5)] * len(reais)
        y_t, y_l, y_a = np.array(reais), np.array(preds_lstm), np.array(preds_arima)
        def calc(yt, yp):
            mask = yt > 1e-6
            mape = float(np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100) if np.any(mask) else 0.0
            return {
                "mae": round(float(mean_absolute_error(yt, yp)), 5),
                "rmse": round(float(np.sqrt(mean_squared_error(yt, yp))), 5),
                "mape": round(mape, 2),
                "r2": round(float(r2_score(yt, yp)), 4) if len(yt) > 1 else 0.0
            }
        return {
            "status": "sucesso",
            "labels": labels[-30:],
            "reais": reais[-30:],
            "lstm": preds_lstm[-30:],
            "arima": preds_arima[-30:],
            "metricas": {"lstm": calc(y_t, y_l), "arima": calc(y_t, y_a)}
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro ao gerar histórico: {str(e)}")

# 1. Instala o localtunnel de forma silenciosa
!npm install -g localtunnel -q

import subprocess
import threading
import time

# 2. Função para iniciar o servidor Uvicorn em background
def rodar_servidor():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
    server = uvicorn.Server(config)
    nest_asyncio.apply()
    import asyncio
    asyncio.run(server.serve())

thread_servidor = threading.Thread(target=rodar_servidor, daemon=True)
thread_servidor.start()
time.sleep(2) # Aguarda o servidor subir

# 3. Inicia o túnel localtunnel na porta 8000
processo_lt = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# 4. Captura e exibe a URL pública gerada
print("\n====================================================")
print("🚀 AGUARDANDO URL PÚBLICA DO LOCALTUNNEL...")
print("====================================================")

for linha in processo_lt.stdout:
    if "url" in linha.lower():
        url_limpa = linha.strip()
        print(f"\n🔗 COPIE ESTA URL E COLE NA DASHBOARD:\n{url_limpa}\n")
        print("====================================================")
        break

Carregando modelo e scalers...
Modelo carregado! Formato: (10 passos, 6 features)
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.2
npm notice To update run: npm install -g npm@12.2.2
npm notice
⠙
🚀 AGUARDANDO URL PÚBLICA DO LOCALTUNNEL...

🔗 COPIE ESTA URL E COLE NA DASHBOARD:
your url is: https://eager-foxes-call.loca.lt

